# Workflow 17: ImageNet and RadImageNet under two training protocols
Run these cells in order using **BRISC Linux (GPU)**. The training cell launches 12 isolated processes sequentially: two pretrained sources, two protocols, three seeds (42, 43, 44). Each process releases GPU memory when it exits.

The notebook uses `configs/resnet_comparison.json` and `scripts/train_resnet_comparison.py`. New outputs go to `runs/`; historical results remain separate. The 678-image test partition is unchanged. This does not resolve the pending pHash review.

The adapted protocol unfreezes the backbone except BatchNorm, with learning rate 1e-4 and up to 30 fine-tuning epochs. The uniform protocol unfreezes the top 60 layers, with learning rate 1e-5 and up to 12 epochs. Both use the same fixed stratified validation subset, head and batch size 8. Validation receives no augmentation. Checkpoint selection uses validation accuracy across both training stages; report every seed, without choosing the best test result.

RadImageNet preprocessing follows the pinned TensorFlow example (Keras caffe preprocessing, then division by 255). It differs from the historical [-1, 1] implementation. The config identifies the local pretrained files by SHA256; it does not independently authenticate their publisher. This extension is exploratory, and the preprocessing/validation changes mean it is not an exact repeat of historical training.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
assert sys.platform.startswith("linux"), "Select the BRISC Linux (GPU) kernel."
ROOT = Path(os.environ.get("BRISC_REPO", "/mnt/c/Users/cbot/Desktop/brain_tumor_brisc_tmp"))
DATASET = Path(os.environ.get("BRISC_DATASET", "/root/brisc/data/brisc2025_clean/classification_task"))
PYTHON = sys.executable
print("Repository:", ROOT, "\nPython:", PYTHON)
subprocess.run([PYTHON, "-c", "import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g, 'GPU missing'"], check=True)


## Prepare the experiment


In [ ]:
from datetime import datetime
RUN_NAME = "resnet_comparison_" + datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "runs" / RUN_NAME
IMAGENET_WEIGHTS = Path("/root/.keras/models/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5")
RADIMAGENET_WEIGHTS = Path("/root/brisc/radimagenet/RadImageNet-ResNet50_notop.h5")
subprocess.run([PYTHON, str(ROOT / "scripts/train_resnet_comparison.py"),
    "--output", str(OUT), "--dataset", str(DATASET),
    "--imagenet-weights", str(IMAGENET_WEIGHTS),
    "--radimagenet-weights", str(RADIMAGENET_WEIGHTS)], check=True, cwd=ROOT)
plan = json.loads((OUT / "plan.json").read_text())
display(pd.DataFrame(plan["jobs"]))
print("Outputs:", OUT)


## Run all 12 training jobs
This cell starts training. Keep the laptop powered and awake. To stop, interrupt this cell; it terminates its active child process. Completed jobs are skipped if you rerun this cell with the same OUT. A partial job is preserved and raises an error; inspect it before preparing a fresh experiment. Logs and per-epoch progress are saved inside OUT.


In [ ]:
for i, job in enumerate(plan["jobs"], 1):
    folder = OUT / job["id"]
    if (folder / "metrics.json").exists():
        print("Already complete:", job["id"])
        continue
    print(f"\n[{i}/{len(plan['jobs'])}] {job['id']}", flush=True)
    env = dict(os.environ, PYTHONHASHSEED=str(job["seed"]), TF_DETERMINISTIC_OPS="1")
    with (OUT / (job["id"] + ".log")).open("a") as log:
        process = subprocess.Popen([PYTHON, "-u", str(ROOT / "scripts/train_resnet_comparison.py"),
            "--output", str(OUT), "--job", job["id"]], cwd=ROOT, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            if process.wait() != 0:
                raise RuntimeError("Training failed; inspect " + str(OUT / (job["id"] + ".log")))
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
print("All planned jobs completed.")


## Results across seeds


In [ ]:
rows = [json.loads(p.read_text()) for p in OUT.glob("*/metrics.json")]
results = pd.DataFrame(rows)
display(results[["origin", "protocol", "seed", "accuracy", "ci95_lo", "ci95_hi", "f1_macro", "auc_macro_ovr"]])
results.to_csv(OUT / "all_results.csv", index=False)
summary = results.groupby(["origin", "protocol"]).agg(
    completed_seeds=("seed", "count"), accuracy_mean=("accuracy", "mean"),
    accuracy_sd=("accuracy", "std"), f1_mean=("f1_macro", "mean"), auc_mean=("auc_macro_ovr", "mean"))
display(summary)
summary.to_csv(OUT / "summary_across_seeds.csv")
print("Per-run bootstrap intervals describe the test sample; across-seed SD describes training variation.")
print("Report every planned configuration. No revised pHash benchmark or external validation was performed.")
